### Bart finetune

In [ ]:
! pip install transformers datasets accelerate

In [ ]:
!pip install scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

preprocessing

In [ ]:
! cd

for later: bart-larger

In [ ]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large")

# def tokenize(example):
#     return tokenizer(example["text"], truncation=True, padding="max_length")

# dataset = dataset.map(tokenize, batched=True)
# dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
!pip install datasets

## DistilBERT multi-feature phishing detection

In [1]:
import torch
import torch.nn as nn
from transformers import DistilBertPreTrainedModel, DistilBertModel

class DistilBertMultiTaskClassifier(DistilBertPreTrainedModel):
    """Multi-head classifier for phishing feature prediction"""
    def __init__(self, config):
        super().__init__(config)
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.dropout)
        
        # One head per feature
        self.intent_classifier = nn.Linear(config.hidden_size, 3)      # normal/phishing/suspicious
        self.manipulation_classifier = nn.Linear(config.hidden_size, 2) # 0 or 1
        self.request_type_classifier = nn.Linear(config.hidden_size, 4) # none/credentials/payment/personal_info
        self.impersonation_classifier = nn.Linear(config.hidden_size, 2) # 0 or 1
    
    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        # Handle labels passed as separate kwargs (for evaluation)
        if labels is None and 'intent' in kwargs:
            labels = {
                'intent': kwargs.pop('intent'),
                'manipulation': kwargs.pop('manipulation'),
                'request_type': kwargs.pop('request_type'),
                'impersonation': kwargs.pop('impersonation'),
            }
        
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs[0][:, 0, :]  # [CLS] token
        pooled_output = self.dropout(pooled_output)
        
        # Multi-head predictions
        intent_logits = self.intent_classifier(pooled_output)
        manipulation_logits = self.manipulation_classifier(pooled_output)
        request_type_logits = self.request_type_classifier(pooled_output)
        impersonation_logits = self.impersonation_classifier(pooled_output)
        
        # Compute loss if labels provided
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = (
                loss_fn(intent_logits, labels["intent"]) +
                loss_fn(manipulation_logits, labels["manipulation"]) +
                loss_fn(request_type_logits, labels["request_type"]) +
                loss_fn(impersonation_logits, labels["impersonation"])
            ) / 4.0  # Average loss across heads
        
        return {
            "loss": loss,
            "intent": intent_logits,
            "manipulation": manipulation_logits,
            "request_type": request_type_logits,
            "impersonation": impersonation_logits,
        }

/mnt/DISK/Studies/SEM 4/Project Course 299/Phishing-detector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

print("loading")
# ========================
# LOAD CSV
# ========================
df = pd.read_csv(r"dataset/processed/emails_sms_combined.csv")

# sanity check
df = df[["channel", "text", "label"]]
df["label"] = df["label"].astype(int)
print(df.head())

# ========================
# CONVERT TO DATASET
# ========================
dataset = Dataset.from_pandas(df)

# ========================
# PREPROCESS (MULTI-FEATURE MAPPING)
# ========================
def heuristic_label_mapping(is_phishing):
    """Map single label to multi-feature labels using heuristics"""
    if is_phishing == 1:
        return {
            "intent": 1,                           # phishing
            "manipulation": 1,
            "request_type": 1,                     # credentials (most common phishing request)
            "impersonation": 1
        }
    else:
        return {
            "intent": 0,                           # normal
            "manipulation": 0,
            "request_type": 0,                     # none
            "impersonation": 0
        }

def preprocess(example):
    labels = heuristic_label_mapping(example['label'])
    
    return {
        "text": example["text"],
        "channel": example["channel"],
        **labels
    }

dataset = dataset.map(preprocess)

# ========================
# TOKENIZER
# ========================
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Reset and remove any stale columns
dataset.reset_format()
cols_to_keep = ["text", "channel", "intent", "manipulation", "request_type", "impersonation"]
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in cols_to_keep])

def tokenize(example):
    text = f"{example['channel']}: {example['text']}"
    return tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=False, keep_in_memory=True)

# ========================
# PREPARE FOR TRAINING
# ========================
# Remove unnecessary columns, but KEEP labels
dataset = dataset.remove_columns([col for col in dataset.column_names 
                                  if col not in ["input_ids", "attention_mask", 
                                                  "intent", "manipulation", 
                                                  "request_type", "impersonation"]])

# ========================
# TRAIN / TEST SPLIT
# ========================
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

# ========================
# LOAD MODEL
# ========================
config = AutoConfig.from_pretrained("distilbert-base-uncased")
model = DistilBertMultiTaskClassifier(config)

# Use CPU to avoid CUDA memory issues (DistilBERT is small enough for CPU)
device = "cpu"
model.to(device)

# ========================
# METRICS
# ========================
def compute_metrics(eval_pred):
    """Compute metrics for multi-task classification"""
    predictions, labels = eval_pred
    
    # predictions is ((logits_dict,),) so extract logits_dict
    logits = predictions[0]
    
    # predictions is a dict with keys: intent, manipulation, request_type, impersonation
    # Each value is logits array of shape (batch_size, num_classes)
    
    intent_preds = logits["intent"].argmax(axis=1)
    manipulation_preds = logits["manipulation"].argmax(axis=1)
    request_type_preds = logits["request_type"].argmax(axis=1)
    impersonation_preds = logits["impersonation"].argmax(axis=1)
    
    intent_labels = labels["intent"]
    manipulation_labels = labels["manipulation"]
    request_type_labels = labels["request_type"]
    impersonation_labels = labels["impersonation"]
    
    return {
        "intent_accuracy": accuracy_score(intent_labels, intent_preds),
        "intent_f1": f1_score(intent_labels, intent_preds, average="weighted", zero_division=0),
        "manipulation_accuracy": accuracy_score(manipulation_labels, manipulation_preds),
        "request_type_accuracy": accuracy_score(request_type_labels, request_type_preds),
        "impersonation_accuracy": accuracy_score(impersonation_labels, impersonation_preds),
    }


loading
  channel                                               text  label
0   email  We detected malware activity on your workstati...      1
1     sms  Critical virus detected on device. Pay [Amount...      1
2   email  Our security team identified your IP participa...      1
3     sms  Ransomware installed. Send [Amount] in BTC to ...      1
4   email  Your website has been injected with remote acc...      1


Map: 100%|██████████| 20985/20985 [00:29<00:00, 722.47 examples/s] 


In [5]:
from transformers import DataCollatorWithPadding

# ========================
# CUSTOM DATA COLLATOR
# ========================
class MultiTaskDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    
    def __call__(self, batch):
        # Separate labels from input data BEFORE processing
        labels = {
            "intent": [],
            "manipulation": [],
            "request_type": [],
            "impersonation": []
        }
        
        # Extract labels while removing them from items
        input_features = []
        for item in batch:
            labels["intent"].append(item.pop("intent"))
            labels["manipulation"].append(item.pop("manipulation"))
            labels["request_type"].append(item.pop("request_type"))
            labels["impersonation"].append(item.pop("impersonation"))
            # Only keep tokenizer columns: input_ids, token_type_ids, attention_mask
            input_features.append(item)
        
        # Collate input features
        batch_data = self.data_collator(input_features)
        
        # Add labels as tensors
        for key in labels:
            batch_data[key] = torch.tensor(labels[key], dtype=torch.long)
        
        return batch_data

# ========================
# CUSTOM TRAINER
# ========================
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Extract labels
        intent_labels = inputs.pop("intent", None)
        manipulation_labels = inputs.pop("manipulation", None)
        request_type_labels = inputs.pop("request_type", None)
        impersonation_labels = inputs.pop("impersonation", None)
        
        # Forward pass
        outputs = model(**inputs)
        
        # Compute loss for each head
        loss_fn = torch.nn.CrossEntropyLoss()
        loss = (
            loss_fn(outputs["intent"], intent_labels) +
            loss_fn(outputs["manipulation"], manipulation_labels) +
            loss_fn(outputs["request_type"], request_type_labels) +
            loss_fn(outputs["impersonation"], impersonation_labels)
        ) / 4.0
        
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        # Extract labels
        labels = {}
        if "intent" in inputs:
            labels["intent"] = inputs.pop("intent")
        if "manipulation" in inputs:
            labels["manipulation"] = inputs.pop("manipulation")
        if "request_type" in inputs:
            labels["request_type"] = inputs.pop("request_type")
        if "impersonation" in inputs:
            labels["impersonation"] = inputs.pop("impersonation")
        
        # Forward pass
        with self.compute_loss_context_manager():
            outputs = model(**inputs)
        
        # Extract loss and logits
        loss = outputs.get("loss")
        logits = {k: v for k, v in outputs.items() if k != "loss"}
        
        return loss, logits, labels

data_collator = MultiTaskDataCollator(tokenizer)

In [60]:
# Debug: Check dataset columns
print("Train dataset columns:", train_dataset.column_names)
print("Train dataset sample:")
print(train_dataset[0])

Train dataset columns: ['intent', 'manipulation', 'request_type', 'impersonation', 'input_ids', 'attention_mask']
Train dataset sample:
{'intent': 1, 'manipulation': 1, 'request_type': 1, 'impersonation': 1, 'input_ids': [101, 10373, 1024, 7192, 2115, 4923, 1010, 10142, 1998, 2005, 2489, 5547, 2115, 7016, 1999, 1017, 2781, 2017, 2024, 4909, 2023, 10373, 2004, 1037, 4942, 29234, 2099, 2000, 1996, 9144, 25974, 4630, 5653, 2075, 2862, 1012, 2000, 6366, 4426, 2013, 2023, 1998, 3141, 10373, 7201, 11562, 2182, 1024, 4895, 6342, 5910, 26775, 20755, 2026, 10373, 2104, 3021, 1006, 1055, 1007, 29029, 2516, 3523, 2011, 1996, 8746, 2149, 3519, 1010, 2566, 2930, 19123, 1010, 20423, 1006, 1037, 1007, 1006, 1016, 1007, 1997, 1055, 1012, 29029, 1010, 1037, 3661, 3685, 2022, 9530, 7363, 6672, 7174, 2860, 12403, 2213, 2065, 1996, 4604, 2121, 2950, 3967, 2592, 1998, 1037, 4118, 1997, 1000, 8208, 1000, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1

In [7]:
# ========================
# TRAINING ARGS
# ========================
training_args = TrainingArguments(
    output_dir="./distilbert-phishing-model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    fp16=False,  # CPU doesn't support mixed precision
    save_strategy="no",
    remove_unused_columns=False,  # <- KEY FIX: Preserve all columns including labels
    use_cpu=True  # Explicitly use CPU to avoid CUDA issues
)

In [8]:
# ========================
# INSTANTIATE TRAINER with loaded model
# ========================
trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Training already completed, skip to evaluation
# print("Starting training...")
# trainer.train()
# print("Training complete!")

# ========================
# EVALUATE
# ========================
try:
    torch.cuda.empty_cache()  # Clear GPU memory before evaluation
    results = trainer.evaluate()
    print("Evaluation results:", results)
except Exception as e:
    print("Evaluation completed but callback failed:", str(e))
    # Try to get results from trainer state
    if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
        last_log = trainer.state.log_history[-1] if trainer.state.log_history else {}
        print("Last log:", last_log)

KeyboardInterrupt: 

In [6]:
# Recreate model with updated forward method
from safetensors.torch import load_file

config = AutoConfig.from_pretrained("distilbert-base-uncased")
model = DistilBertMultiTaskClassifier(config)
model.to(device)
state_dict = load_file("./distilbert-phishing-model/model.safetensors")
model.load_state_dict(state_dict)
model.eval()

DistilBertMultiTaskClassifier(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
      

In [7]:
model.save_pretrained("./distilbert-phishing-model")
tokenizer.save_pretrained("./distilbert-phishing-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]


('./distilbert-phishing-model/tokenizer_config.json',
 './distilbert-phishing-model/tokenizer.json')

# load back

In [8]:
# ========================
# REASONING LAYER
# ========================

def compute_phishing_score(manipulation, request_type, impersonation):
    """
    Rule-based scoring function to compute final phishing risk
    
    Scoring rules:
    - manipulation=1: +2
    - request_type=credentials(1): +3
    - request_type=payment(2): +2
    - request_type=personal_info(3): +1
    - impersonation=1: +2
    
    Threshold: score >= 4 → phishing
    """
    score = 0
    
    if manipulation == 1:
        score += 2
    
    if request_type == 1:  # credentials
        score += 3
    elif request_type == 2:  # payment
        score += 2
    elif request_type == 3:  # personal_info
        score += 1
    
    if impersonation == 1:
        score += 2
    
    return score

def get_final_label(score):
    """Convert score to binary label"""
    return 1 if score >= 4 else 0

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file

model_path = "./distilbert-phishing-model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)
model = DistilBertMultiTaskClassifier(config)
state_dict = load_file(f"{model_path}/model.safetensors")
model.load_state_dict(state_dict)

# Use CPU to avoid CUDA memory issues
device = "cpu"
model.to(device)
model.eval()

DistilBertMultiTaskClassifier(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
      

In [10]:
# Update the forward method to handle kwargs
def updated_forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
    # Handle labels passed as separate kwargs (for evaluation)
    if labels is None and 'intent' in kwargs:
        labels = {
            'intent': kwargs.pop('intent'),
            'manipulation': kwargs.pop('manipulation'),
            'request_type': kwargs.pop('request_type'),
            'impersonation': kwargs.pop('impersonation'),
        }
    
    outputs = self.distilbert(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    pooled_output = outputs[0][:, 0, :]  # [CLS] token
    pooled_output = self.dropout(pooled_output)
    
    # Multi-head predictions
    intent_logits = self.intent_classifier(pooled_output)
    manipulation_logits = self.manipulation_classifier(pooled_output)
    request_type_logits = self.request_type_classifier(pooled_output)
    impersonation_logits = self.impersonation_classifier(pooled_output)
    
    # Compute loss if labels provided
    loss = None
    if labels is not None:
        loss_fn = nn.CrossEntropyLoss()
        loss = (
            loss_fn(intent_logits, labels["intent"]) +
            loss_fn(manipulation_logits, labels["manipulation"]) +
            loss_fn(request_type_logits, labels["request_type"]) +
            loss_fn(impersonation_logits, labels["impersonation"])
        ) / 4.0  # Average loss across heads
    
    return {
        "loss": loss,
        "intent": intent_logits,
        "manipulation": manipulation_logits,
        "request_type": request_type_logits,
        "impersonation": impersonation_logits,
    }

# Assign the updated forward method
model.forward = updated_forward.__get__(model, DistilBertMultiTaskClassifier)

In [ ]:
import torch

# Label mappings
INTENT_LABELS = {0: "normal", 1: "phishing", 2: "suspicious"}
REQUEST_TYPE_LABELS = {0: "none", 1: "credentials", 2: "payment", 3: "personal_info"}
BINARY_LABELS = {0: "no", 1: "yes"}

def analyze_message(text, channel="email"):
    """
    Comprehensive phishing analysis with multi-feature prediction
    
    Args:
        text: Message content to analyze
        channel: 'email' or 'sms'
    
    Returns:
        Dictionary with predictions and final label
    """
    # Prepare input
    combined_text = f"{channel}: {text}"
    inputs = tokenizer(
        combined_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )
    
    # Move to device (CPU)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract predictions
    intent_logits = outputs["intent"]
    manipulation_logits = outputs["manipulation"]
    request_type_logits = outputs["request_type"]
    impersonation_logits = outputs["impersonation"]
    
    # Get class predictions
    intent_idx = intent_logits.argmax(dim=1).item()
    manipulation_idx = manipulation_logits.argmax(dim=1).item()
    request_type_idx = request_type_logits.argmax(dim=1).item()
    impersonation_idx = impersonation_logits.argmax(dim=1).item()
    
    # Get confidence scores
    intent_confidence = torch.softmax(intent_logits, dim=1)[0][intent_idx].item()
    manipulation_confidence = torch.softmax(manipulation_logits, dim=1)[0][manipulation_idx].item()
    request_type_confidence = torch.softmax(request_type_logits, dim=1)[0][request_type_idx].item()
    impersonation_confidence = torch.softmax(impersonation_logits, dim=1)[0][impersonation_idx].item()
    
    # Compute phishing score
    score = compute_phishing_score(manipulation_idx, request_type_idx, impersonation_idx)
    final_label = get_final_label(score)
    
    return {
        "text": text[:100] + "..." if len(text) > 100 else text,
        "channel": channel,
        "intent": {
            "prediction": INTENT_LABELS[intent_idx],
            "confidence": intent_confidence
        },
        "manipulation": {
            "prediction": BINARY_LABELS[manipulation_idx],
            "confidence": manipulation_confidence
        },
        "request_type": {
            "prediction": REQUEST_TYPE_LABELS[request_type_idx],
            "confidence": request_type_confidence
        },
        "impersonation": {
            "prediction": BINARY_LABELS[impersonation_idx],
            "confidence": impersonation_confidence
        },
        "score": score,
        "final_label": "phishing" if final_label == 1 else "legitimate"
    }

# Example usage
print("=" * 60)
print("PHISHING DETECTION ANALYSIS")
print("=" * 60)

test_messages = [
    ("Your balance limit has been reached", "email"),
    ("Please click on the following link to verify your identity", "sms"),
    ("Confirm your banking credentials now to avoid suspension", "email"),
]

for msg, ch in test_messages:
    result = analyze_message(msg, channel=ch)
    print(f"\nMessage: {result['text']}")
    print(f"Channel: {result['channel']}")
    print(f"Intent: {result['intent']['prediction']} (conf: {result['intent']['confidence']:.2f})")
    print(f"Manipulation: {result['manipulation']['prediction']} (conf: {result['manipulation']['confidence']:.2f})")
    print(f"Request Type: {result['request_type']['prediction']} (conf: {result['request_type']['confidence']:.2f})")
    print(f"Impersonation: {result['impersonation']['prediction']} (conf: {result['impersonation']['confidence']:.2f})")
    print(f"Risk Score: {result['score']}/7")
    print(f"Final Label: {result['final_label'].upper()}")
    print("-" * 60)


PHISHING DETECTION ANALYSIS

Message: Your balance limit has been reached
Channel: email
Intent: suspicious (conf: 0.38)
Manipulation: yes (conf: 0.54)
Request Type: payment (conf: 0.40)
Impersonation: no (conf: 0.65)
Risk Score: 4/7
Final Label: PHISHING
------------------------------------------------------------

Message: Please click on the following link to verify your identity
Channel: sms
Intent: suspicious (conf: 0.37)
Manipulation: yes (conf: 0.59)
Request Type: payment (conf: 0.41)
Impersonation: no (conf: 0.70)
Risk Score: 4/7
Final Label: PHISHING
------------------------------------------------------------

Message: Confirm your banking credentials now to avoid suspension
Channel: email
Intent: suspicious (conf: 0.36)
Manipulation: yes (conf: 0.57)
Request Type: payment (conf: 0.47)
Impersonation: no (conf: 0.69)
Risk Score: 4/7
Final Label: PHISHING
------------------------------------------------------------
